In [1]:
import joblib
import numpy as np
import requests
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(model, "model.joblib")
joblib.dump(iris.target_names, "target_names.joblib") 

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



['target_names.joblib']

In [3]:
loaded_model = joblib.load("model.joblib")
loaded_predictions = loaded_model.predict(X_test)
print("Predictions identical:", np.array_equal(y_pred, loaded_predictions))

Predictions identical: True


In [4]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model = joblib.load("model.joblib")
target_names = joblib.load("target_names.joblib")

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"}), 200

def validate_features(features):
    if not isinstance(features, list):
        return "Features must be a list."
    if len(features) != 4:
        return "Features must contain exactly 4 values."
    for v in features:
        if not isinstance(v, (int, float)) or isinstance(v, bool):
            return "All feature values must be numeric."
    return None

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)
    if not data or "features" not in data:
        return jsonify({"error": "Missing 'features' key"}), 400
    features = data["features"]
    error = validate_features(features)
    if error:
        return jsonify({"error": error}), 400
    X = np.array(features).reshape(1, -1)
    pred = model.predict(X)[0]
    probs = model.predict_proba(X)[0]
    return jsonify({
        "predicted_class": str(target_names[pred]),
        "probabilities": {
            str(name): round(float(p), 4)
            for name, p in zip(target_names, probs)
        }
    })

@app.route("/predict_batch", methods=["POST"])
def predict_batch():
    data = request.get_json(silent=True)
    if not data or "samples" not in data:
        return jsonify({"error": "Missing 'samples' key"}), 400
    samples = data["samples"]
    if not isinstance(samples, list):
        return jsonify({"error": "'samples' must be a list"}), 400
    results = []
    for sample in samples:
        error = validate_features(sample)
        if error:
            return jsonify({"error": error}), 400
        X = np.array(sample).reshape(1, -1)
        pred = model.predict(X)[0]
        probs = model.predict_proba(X)[0]
        results.append({
            "predicted_class": str(target_names[pred]),
            "probabilities": {
                str(name): round(float(p), 4)
                for name, p in zip(target_names, probs)
            }
        })
    return jsonify({"predictions": results})

if __name__ == "__main__":
    app.run(debug=False, port=5000)

Overwriting app.py


In [6]:
response = requests.get("http://127.0.0.1:5000/health")
print("Status Code:", response.status_code)
print("Response:", response.json()) 

Status Code: 200
Response: {'status': 'healthy'}


In [7]:
sample = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post("http://127.0.0.1:5000/predict", json=sample)
print("Status Code:", response.status_code)
print("Result:", response.json())

Status Code: 200
Result: {'predicted_class': 'setosa', 'probabilities': {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}}


In [8]:
sample = {"features": [5.1, 3.5, 1.4, 0.2]}
response = requests.post("http://127.0.0.1:5000/predict", json=sample)
print("Status Code:", response.status_code)
print("Raw response:")
print(response.text[:1000])

Status Code: 200
Raw response:
{"predicted_class":"setosa","probabilities":{"setosa":1.0,"versicolor":0.0,"virginica":0.0}}



In [9]:
# Missing 'features' key
bad1 = {}
r1 = requests.post("http://127.0.0.1:5000/predict", json=bad1)
print(r1.status_code, r1.json())

# Wrong number of features
bad2 = {"features": [5.1, 3.5, 1.4]}
r2 = requests.post("http://127.0.0.1:5000/predict", json=bad2)
print(r2.status_code, r2.json())

# Non-numeric values
bad3 = {"features": ["a", "b", "c", "d"]}
r3 = requests.post("http://127.0.0.1:5000/predict", json=bad3)
print(r3.status_code, r3.json()) 

400 {'error': "Missing 'features' key"}
400 {'error': 'Features must contain exactly 4 values.'}
400 {'error': 'All feature values must be numeric.'}


In [10]:
samples = X_test[:5].tolist()
response = requests.post(
    "http://127.0.0.1:5000/predict_batch",
    json={"samples": samples}
)
api_results = response.json()
print(api_results)

{'predictions': [{'predicted_class': 'versicolor', 'probabilities': {'setosa': 0.0, 'versicolor': 0.99, 'virginica': 0.01}}, {'predicted_class': 'setosa', 'probabilities': {'setosa': 1.0, 'versicolor': 0.0, 'virginica': 0.0}}, {'predicted_class': 'virginica', 'probabilities': {'setosa': 0.0, 'versicolor': 0.03, 'virginica': 0.97}}, {'predicted_class': 'versicolor', 'probabilities': {'setosa': 0.0, 'versicolor': 1.0, 'virginica': 0.0}}, {'predicted_class': 'versicolor', 'probabilities': {'setosa': 0.0, 'versicolor': 0.87, 'virginica': 0.13}}]}


In [11]:
local_preds = model.predict(X_test[:5])
local_classes = [iris.target_names[i] for i in local_preds]
api_classes = [p["predicted_class"] for p in api_results["predictions"]]
print("Local:", local_classes)
print("API:  ", api_classes)
print("Match:", local_classes == api_classes)

Local: [np.str_('versicolor'), np.str_('setosa'), np.str_('virginica'), np.str_('versicolor'), np.str_('versicolor')]
API:   ['versicolor', 'setosa', 'virginica', 'versicolor', 'versicolor']
Match: True


In [12]:
import os
os.makedirs("deployment", exist_ok=True)

In [13]:
%%writefile deployment/README.md
# Iris Classifier API

## What the Model Does

This service exposes a trained **Random Forest classifier** (100 trees) that predicts the species of an Iris flower from four numeric measurements: sepal length, sepal width, petal length, and petal width (all in centimeters). The model was trained on the classic Iris dataset using an 80/20 train/test split and achieves high accuracy on the held-out test set. It returns one of three species — *setosa*, *versicolor*, or *virginica* — along with the predicted probability for each class. The service is implemented as a Flask HTTP API that loads the serialized model at startup and serves predictions over JSON.

## How to Run

1. Make sure you have the required dependencies installed:

Writing deployment/README.md


## Reflection

### Production Deployment

The current setup uses Flask's built-in development server with `app.run()`, which is single-threaded, lacks proper request queuing, and is explicitly not recommended for production. For a real deployment, several changes would be needed:

- **WSGI server**: Replace `app.run()` with a production-grade WSGI server such as **Gunicorn** or **uWSGI**, running multiple worker processes behind it (e.g., `gunicorn -w 4 -b 0.0.0.0:5000 app:app`). This handles concurrent requests properly and recovers from worker crashes.
- **Reverse proxy**: Put **Nginx** in front of Gunicorn to handle TLS termination, static files, request buffering, and rate limiting.
- **HTTPS**: Serve everything over HTTPS using TLS certificates (e.g., via Let's Encrypt) to encrypt traffic.
- **Containerization**: Package the app and its dependencies in a **Docker image** so it runs identically on any host. The image would include the Python runtime, dependencies, the model artifacts, and the app code, with a defined entrypoint that starts Gunicorn.
- **Environment variables**: Move configuration (port, model path, log level, allowed hosts) out of the code and into environment variables, so the same image can be deployed across dev/staging/prod without rebuilds.
- **Secrets management**: Use a secrets store (AWS Secrets Manager, HashiCorp Vault) rather than hardcoding any API keys or credentials.
- **Health and readiness probes**: The existing `/health` endpoint already supports load balancer health checks; in Kubernetes, this would be wired into liveness and readiness probes.

### Model Versioning

A versioning strategy is essential for safe iteration:

- **Artifact versioning**: Save each trained model with a version tag and metadata (e.g., `model_v1.2.0.joblib`) and track them in a model registry such as **MLflow** or **Weights & Biases**. The registry records the training data hash, hyperparameters, evaluation metrics, and a timestamp.
- **API versioning**: Include the model version in the response payload so clients know which model produced a given prediction.
- **Staged rollout**: When a new model is trained, validate it against a holdout set and historical production traffic. Only promote it after it meets a minimum quality bar. Use **shadow deployments** (run new model alongside old, compare predictions, but only serve the old one) or **canary releases** (route a small percentage of traffic to the new model and monitor) to catch regressions before full rollout.
- **Rollback**: Keep the previous model artifact deployable at any time, so if the new model degrades real-world metrics, switching back is one config change away.

### Monitoring

Several signals would be tracked:

- **Operational metrics**: request rate, prediction latency (p50, p95, p99), error rate, CPU and memory usage. Exported to **Prometheus** and visualized in **Grafana**.
- **Input drift**: log the input feature distributions and compare against the training distribution using statistical tests (KS test, PSI). A drift alert means production data has shifted and the model may be making predictions in a regime it wasn't trained on.
- **Prediction drift**: track the distribution of predicted classes over time. A sudden change in the proportion of `setosa` vs `virginica` predictions can signal upstream data issues.
- **Logging**: log every request with its features, prediction, latency, and model version (with PII handled carefully). Aggregate logs in a system like **ELK** or **Datadog** for debugging.
- **Alerting**: page on-call when error rate exceeds a threshold, latency spikes, or drift metrics breach limits.
- **Ground truth tracking**: where possible, capture eventual labels and compute live accuracy. This is the only direct signal that the model is still doing its job.

### Scaling to 1,000 RPS

The current single-process Flask dev server would not handle this. Scaling architecture:

- **Horizontal scaling**: run many stateless replicas of the API container behind a **load balancer** (e.g., AWS ALB, Kubernetes Service). The model is small and predictions are independent, so requests parallelize trivially.
- **Autoscaling**: configure the orchestrator (Kubernetes HPA, ECS, or similar) to add or remove replicas based on CPU usage and request queue depth.
- **Multiple workers per replica**: each replica runs Gunicorn with several workers, ideally `2 × CPU_cores + 1`, so each replica saturates its CPU.
- **Caching**: if many requests have identical inputs (rare for continuous features but common in some domains), add a thin **Redis** cache in front of the model.
- **Batch inference**: when client throughput is high, group nearby requests into mini-batches inside the worker; scikit-learn handles vectorized prediction far more efficiently than one-at-a-time calls. This is the role of `/predict_batch`, but can also be done transparently with a request coalescing layer.
- **Optimized runtime**: for higher throughput needs, the model can be exported to **ONNX** and served with **ONNX Runtime**, or with a dedicated server like **BentoML**, **TorchServe**, or **NVIDIA Triton**.
- **Async I/O**: switch the framework to **FastAPI** with an ASGI server (Uvicorn), which handles concurrent requests more efficiently than synchronous Flask, especially under high I/O.
- **Edge concerns**: rate limiting, request authentication, and CDN-level caching for the static parts of the response — all handled at the load balancer or API gateway layer.
- **Capacity planning**: load-test the system with tools like **k6** or **Locust** to find the actual ceiling per replica and provision accordingly, with headroom for traffic spikes.